### Search Engine With Tools And Agents

In [1]:
pip install langchain langchain-community langchain-huggingface sentence-transformers faiss-cpu arxiv wikipedia langchain-groq python-dotenv langsmith

Note: you may need to restart the kernel to use updated packages.


In [2]:
# -----------------------------
# Load Environment Variables
# -----------------------------
from dotenv import load_dotenv
import os

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

# -----------------------------
# Built-in Tools
# -----------------------------
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper

# Wikipedia Tool
api_wrapper_wiki = WikipediaAPIWrapper(
    top_k_results=1,
    doc_content_chars_max=250
)

wiki = WikipediaQueryRun(api_wrapper=api_wrapper_wiki)

# Arxiv Tool
api_wrapper_arxiv = ArxivAPIWrapper(
    top_k_results=1,
    doc_content_chars_max=250
)

arxiv = ArxivQueryRun(api_wrapper=api_wrapper_arxiv)

print(wiki.name)
print(arxiv.name)



wikipedia
arxiv


In [ ]:
# -----------------------------
# RAG Tool (Custom Retriever)
# -----------------------------
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
import langchain_community.tools

loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs = loader.load()

documents = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
).split_documents(docs)

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

vectordb = FAISS.from_documents(documents, embeddings)

retriever = vectordb.as_retriever()

from langchain_core.tools import Tool

def retrieve_langsmith_docs(query: str) -> str:
    """Search for information about LangSmith"""
    results = retriever.invoke(query)
    return "\n".join([doc.page_content for doc in results])

retriever_tool = Tool(
    name="langsmith-search",
    func=retrieve_langsmith_docs,
    description="Search any information about LangSmith"
)

# -----------------------------
# Combine Tools
# -----------------------------
tools = [wiki, arxiv, retriever_tool]

# -----------------------------
# LLM (Groq Llama3)
# -----------------------------
from langchain_groq import ChatGroq

if groq_api_key:
    llm = ChatGroq(
        groq_api_key=groq_api_key,
        model_name="llama-3.1-8b-instant"
    )

    from langchain_classic.agents import initialize_agent, AgentType
    
    # Create agent using ZERO_SHOT_REACT_DESCRIPTION
    agent = initialize_agent(
        tools=tools,
        llm=llm,
        agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
        verbose=True,
        max_iterations=10,
        handle_parsing_errors=True
    )

    # Run Queries
    print("="*50)
    print("Query 1: Tell me about LangSmith")
    print("="*50)
    result1 = agent.run("Tell me about LangSmith")
    print(result1)
    
    print("\n" + "="*50)
    print("Query 2: What is machine learning?")
    print("="*50)
    result2 = agent.run("What is machine learning?")
    print(result2)
    
    print("\n" + "="*50)
    print("Query 3: What's the paper 1706.03762 about?")
    print("="*50)
    result3 = agent.run("What's the paper 1706.03762 about?")
    print(result3)
else:
    print("GROQ_API_KEY not found. Please set the GROQ_API_KEY environment variable in .env file to run the agent.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2260.57it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\vishr\AppData\Local\Temp\ipykernel_7564\1719786365.py:58: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the [LangGraph documentation](https://langchain-ai.github.io/langgraph/) as well as guides for [Migrating from AgentExecutor](https://python.langchain.com/docs/how_to/migrate_agent/) and LangGraph's [Pre-bui

Query 1: Tell me about LangSmith


> Entering new AgentExecutor chain...
Thought: I need to find more information about LangSmith to answer the question.
Action: langsmith-search
Action Input: query="LangSmith"
Observation: LangSmith docs - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIToolsPolly AI assistantBetaCLISkillsAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageLangSmith is a framework-agnostic platform for developing, debugging, and deploying AI agents and LLM applications.
It helps you trace requests, evaluate outputs, test prompts, and manage deployments 